# RAG

- LLM이 답변을 생성할 때 참고할 수 있는 자료를 제공해서 답변의 정확도를 높이는 기술

## Naive RAG

- LLM에게 질문과 함께 관련 문서를 제공
- LangChain을 사용하면 chain을 생성할 때 prompt에 관련 문서를 추가해서 LLM에게 전달

### 어떤 문서를 제공할 것인가?

- 단순히 내가 가진 문서를 줄 수도 있지만, 매번 관련 문서를 찾아서 제공하는 것은 비효율적
- 따라서 미리 문서를 처리해서 저장해두고, 질문이 오면 그 질문과 가장 관련성이 높은 문서를 찾아서 제공

### 그럼 문서를 어떻게 처리해서 저장해둘 것인가?

- SQL DB는 정형화된 데이터를 저장하기에 적합하나, 문서와 같이 비정형 데이터를 저장하기에는 적합하지 않음
- 또한, 입력된 질문과 가장 관련성이 높은 문서를 찾기 위한 방법이 필요
- 이를 해결하기 위해 문서를 임베딩 벡터로 변환해서 저장하는 벡터 DB를 사용
- 입력된 질문도 임베딩 벡터로 변환하면 벡터 DB에서 유사한 벡터를 가진 문서를 찾을 수 있음

> 1. 문서 등의 비정형 데이터를 불러와 임베딩 벡터로 변환하여 벡터 DB에 저장
> 2. 사용자가 질문을 하면, 질문을 임베딩 벡터로 변환하여 벡터 DB에서 유사한 벡터를 가진 문서를 찾는 탐색기(Retriever)
> 3. LLM에게 전달하는 prompt에 검색된 문서를 추가: {context}
> 4. LLM은 입력된 질문과 검색된 {context}를 참고하여 답변 생성

## 1. 비정형 데이터를 벡터 DB에 저장하기

### 1-1. 저장할 비정형 데이터 불러오기: Loader

- LangChain에서 제공하는 DocumentLoader를 사용하면 다양한 형식의 데이터를 불러올 수 있다.
- 다양한 소스로부터 불러온 데이터는 `Document` 객체로 변환하여 다룬다.

#### Document 객체

In [33]:
from langchain_core.documents import Document

# page_content: 문서 내용
# metadata: 문서의 메타 데이터 정보를 담고 있는 딕셔너리
doc = Document(page_content="Hello World", metadata={"source": "https://example.com"})
doc

Document(metadata={'source': 'https://example.com'}, page_content='Hello World')

In [11]:
# Document 객체의 데이터는 .연산자로 접근 가능
print('page_content:', doc.page_content)
print('metadata:', doc.metadata)

page_content: Hello World
metadata: {'source': 'https://example.com'}


#### TextLoader: 텍스트 파일 -> List[Document]

In [16]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("./data/rag-keywords.txt", encoding='utf-8')
documents = loader.load()

for i, doc in enumerate(documents):
    print(f"Document {i+1}:")
    print(f"   - 타입: {type(doc)}")
    print(f"   - metadata 타입: {type(doc.metadata)}")
    print(f"   - metadata 내용: {doc.metadata}")
    print(f"   - page_content 타입: {type(doc.page_content)}")
    print(f"   - 내용 길이: {len(doc.page_content)} 문자")
    print(f"   - 내용: \n{doc.page_content[:100]}...")

Document 1:
   - 타입: <class 'langchain_core.documents.base.Document'>
   - metadata 타입: <class 'dict'>
   - metadata 내용: {'source': './data/rag-keywords.txt'}
   - page_content 타입: <class 'str'>
   - 내용 길이: 5733 문자
   - 내용: 
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태...


#### JSONLoader: json 파일 -> List[Document]
- json으로 구조화된 파일을 읽어 자연스러운 문자열로 가져옴
- `jq_schema` 인자에 작성된 정보를 기준으로 `Document` 생성

In [21]:
import json
from pprint import pprint 

file_path = "./data/sokcho_travel_guide.json"
data = json.load(open(file_path, "r", encoding="utf-8"))

pprint(data)

{'date': '2025-07-15',
 'local_cuisine': [{'description': '속초의 명물! 달콤하고 바삭한 맛이 특징', 'dish': '닭강정'},
                   {'description': '오징어 안에 찹쌀과 채소를 넣어 찐 전통 음식',
                    'dish': '오징어순대'}],
 'location': '강원특별자치도 속초시',
 'tourist_attractions': [{'description': '아름다운 해변과 시원한 바닷바람이 매력적인 명소',
                          'name': '속초해수욕장',
                          'tip': '여름철 피서지로 인기 많음'},
                         {'description': '울산바위, 권금성 등 다양한 명소가 있는 국립공원',
                          'name': '설악산 국립공원',
                          'tip': '트레킹과 자연 풍경 감상에 최적'},
                         {'description': '6.25 전쟁 당시 피난민이 정착한 전통 마을',
                          'name': '아바이마을',
                          'tip': '갯배 체험과 향토 음식 즐기기 가능'}],
 'travel_tips': ['해수욕장과 산을 함께 즐길 수 있는 복합형 여행지',
                 '중앙시장 등지에서 지역 음식을 현지 스타일로 맛볼 수 있음',
                 '아바이마을 갯배 체험은 필수 코스'],
 'weather': {'condition': '맑음', 'humidity': '60%', 'temperature': '25°C'}}


In [ ]:
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(
    file_path='./data/sokcho_travel_guide.json',
    # .{$변수명}[] : '변수명'에 해당하는 키의 값을 불러와 해당 위치에 있는 배열에서 각 요소 반복
    # | : 앞의 결과를 뒤에 전달
    # \(.${변수명}) : '변수명'에 해당하는 필드에 저장된 값을 가져오는 포매팅
    jq_schema=r'.tourist_attractions[] | "\(.name): \(.description) (팁: \(.tip))"',
    text_content=False
)

documents = loader.load()
documents

[Document(metadata={'source': 'C:\\dev\\project\\JobFit\\study\\data\\sokcho_travel_guide.json', 'seq_num': 1}, page_content='속초해수욕장: 아름다운 해변과 시원한 바닷바람이 매력적인 명소 (팁: 여름철 피서지로 인기 많음)'),
 Document(metadata={'source': 'C:\\dev\\project\\JobFit\\study\\data\\sokcho_travel_guide.json', 'seq_num': 2}, page_content='설악산 국립공원: 울산바위, 권금성 등 다양한 명소가 있는 국립공원 (팁: 트레킹과 자연 풍경 감상에 최적)'),
 Document(metadata={'source': 'C:\\dev\\project\\JobFit\\study\\data\\sokcho_travel_guide.json', 'seq_num': 3}, page_content='아바이마을: 6.25 전쟁 당시 피난민이 정착한 전통 마을 (팁: 갯배 체험과 향토 음식 즐기기 가능)')]

In [ ]:
loader = JSONLoader(
    file_path='./data/sokcho_travel_guide.json',
    jq_schema=r'.', # '.' 은 그냥 통째로 리턴
    text_content=False
)

documents = loader.load()
documents

[Document(metadata={'source': 'C:\\dev\\project\\JobFit\\study\\data\\sokcho_travel_guide.json', 'seq_num': 1}, page_content='{"location": "\\uac15\\uc6d0\\ud2b9\\ubcc4\\uc790\\uce58\\ub3c4 \\uc18d\\ucd08\\uc2dc", "date": "2025-07-15", "weather": {"temperature": "25\\u00b0C", "condition": "\\ub9d1\\uc74c", "humidity": "60%"}, "tourist_attractions": [{"name": "\\uc18d\\ucd08\\ud574\\uc218\\uc695\\uc7a5", "description": "\\uc544\\ub984\\ub2e4\\uc6b4 \\ud574\\ubcc0\\uacfc \\uc2dc\\uc6d0\\ud55c \\ubc14\\ub2f7\\ubc14\\ub78c\\uc774 \\ub9e4\\ub825\\uc801\\uc778 \\uba85\\uc18c", "tip": "\\uc5ec\\ub984\\ucca0 \\ud53c\\uc11c\\uc9c0\\ub85c \\uc778\\uae30 \\ub9ce\\uc74c"}, {"name": "\\uc124\\uc545\\uc0b0 \\uad6d\\ub9bd\\uacf5\\uc6d0", "description": "\\uc6b8\\uc0b0\\ubc14\\uc704, \\uad8c\\uae08\\uc131 \\ub4f1 \\ub2e4\\uc591\\ud55c \\uba85\\uc18c\\uac00 \\uc788\\ub294 \\uad6d\\ub9bd\\uacf5\\uc6d0", "tip": "\\ud2b8\\ub808\\ud0b9\\uacfc \\uc790\\uc5f0 \\ud48d\\uacbd \\uac10\\uc0c1\\uc5d0 \\ucd5c\\uc8

#### CSVLoader

- csv 파일의 각 행을 `Document`로 변환
- `source_column` 인자에 `page_content`로 쓸 컬럼명 지정; 지정 안하면 전부 다
- `csv_args`: 열 구분자 지정 `{'delimiter': ','}`

In [35]:
import pandas as pd

file_path = "./data/samsung_stock.csv"

# pandas를 이용하여 csv 파일 로드 
df = pd.read_csv(file_path)

print(f"(전체 데이터 수, 전체 컬럼 수): {df.shape}")
df.head()

(전체 데이터 수, 전체 컬럼 수): (129, 7)


,Date,Open,High,Low,Close,Volume,Change
0,2025-01-02,52700,53600,52300,53400,16630538,0.003759
1,2025-01-03,52800,55100,52800,54400,19318046,0.018727
2,2025-01-06,54400,56200,54300,55900,19034284,0.027574
3,2025-01-07,56800,57300,55400,55400,17030235,-0.008945
4,2025-01-08,54800,57500,54700,57300,26593553,0.034296


In [48]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(
    file_path=file_path,
    encoding='utf-8',

    # CSV 파싱 설정
    csv_args={'delimiter': ','} # CSV의 열 구분자 (기본값은 콤마)
)
docs1 = loader.load() 

for i, doc in enumerate(docs1[:2]):
    print("="*50)
    print(f"Document {i+1}:")
    print(f"   - 타입: {type(doc)}")
    print(f"   - page_content 타입: {type(doc.page_content)}")
    print(f"   - metadata 타입: {type(doc.metadata)}")
    print(f"   - metadata 내용: {doc.metadata}")
    print(f"   - 내용 길이: {len(doc.page_content)} 문자")
    print(f"   - 내용: \n{doc.page_content}")
    print()

Document 1:
   - 타입: <class 'langchain_core.documents.base.Document'>
   - page_content 타입: <class 'str'>
   - metadata 타입: <class 'dict'>
   - metadata 내용: {'source': './data/samsung_stock.csv', 'row': 0}
   - 내용 길이: 110 문자
   - 내용: 
Date: 2025-01-02
Open: 52700
High: 53600
Low: 52300
Close: 53400
Volume: 16630538
Change: 0.003759398496240518

Document 2:
   - 타입: <class 'langchain_core.documents.base.Document'>
   - page_content 타입: <class 'str'>
   - metadata 타입: <class 'dict'>
   - metadata 내용: {'source': './data/samsung_stock.csv', 'row': 1}
   - 내용 길이: 109 문자
   - 내용: 
Date: 2025-01-03
Open: 52800
High: 55100
Low: 52800
Close: 54400
Volume: 19318046
Change: 0.01872659176029967



#### WebBaseLoader
- URL의 텍스트를 가져와 `Document`로 변환
- `web_path` 인자에 리스트를 전달하면 여러 페이지를 변환할 수 있음
- `bs_kwargs` 인자를 통해 파서를 지정하면 웹페이지에서 원하는 정보만 가져올 수 있음

In [55]:
import warnings

# 모든 경고 무시
warnings.filterwarnings("ignore")

from langchain_community.document_loaders import WebBaseLoader

# WebBaseLoader를 이용하여 웹 페이지의 HTML을 로드하는 로더 생성
loader = WebBaseLoader(
    web_path="https://www.example.com/",   # 수집할 웹 페이지 URL
    header_template={
        # 서버가 차단하지 않도록 브라우저처럼 보이게 User-Agent 헤더 설정
        "user-agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
    },
    verify_ssl=False,                       # SSL 인증서 검증 비활성화 (테스트용, 실서비스에서는 비권장)
    requests_kwargs={"timeout": 10}         # 요청 타임아웃 10초 설정
)

docs = loader.load()

print(f"로드된 문서의 수: {len(docs)}")
print(f"첫번째 문서의 메타정보 확인:\n {docs[0].metadata}")
print(f"첫번째 문서의 데이터 확인:\n {docs[0].page_content}")

로드된 문서의 수: 1
첫번째 문서의 메타정보 확인:
 {'source': 'https://www.example.com/', 'title': 'Example Domain', 'language': 'en'}
첫번째 문서의 데이터 확인:
 Example DomainExample DomainThis domain is for use in documentation examples without needing permission. Avoid use in operations.Learn more



In [61]:
# WebBaseLoader를 이용하여 여러 웹 페이지를 한 번에 로드하는 로더 생성
loader = WebBaseLoader(
    web_paths=["https://www.example.com/", "https://google.com"],  # 수집할 여러 웹 페이지 URL 리스트
    verify_ssl=False
)

# 지정된 URL들에서 HTML을 가져와 LangChain Document 형태로 로드
docs = loader.load()

# 로드된 Document 객체의 개수 출력
print(f"로드된 문서의 수: {len(docs)}")

for i, doc in enumerate(docs):
    print("="*50)
    print(f"Document {i+1}:")
    print(f"   - 타입: {type(doc)}")
    print(f"   - page_content 타입: {type(doc.page_content)}")
    print(f"   - metadata 타입: {type(doc.metadata)}")
    print(f"   - metadata 내용: {doc.metadata}")
    print(f"   - 내용 길이: {len(doc.page_content)} 문자")
    print()

로드된 문서의 수: 2
Document 1:
   - 타입: <class 'langchain_core.documents.base.Document'>
   - page_content 타입: <class 'str'>
   - metadata 타입: <class 'dict'>
   - metadata 내용: {'source': 'https://www.example.com/', 'title': 'Example Domain', 'language': 'en'}
   - 내용 길이: 140 문자

Document 2:
   - 타입: <class 'langchain_core.documents.base.Document'>
   - page_content 타입: <class 'str'>
   - metadata 타입: <class 'dict'>
   - metadata 내용: {'source': 'https://google.com', 'title': 'Google', 'language': 'ko'}
   - 내용 길이: 106 문자



In [63]:
from langchain_community.document_loaders import WebBaseLoader
from bs4 import SoupStrainer

# 네이트 판 댓글만 추출하기 위한 WebBaseLoader 설정
loader = WebBaseLoader(
    # 가져올 웹 페이지 주소 (네이트 판 특정 게시글)
    web_path="https://pann.nate.com/talk/350939697",
    
    # HTTP 요청 시 사용할 헤더 설정 (브라우저처럼 보이도록)
    header_template={
        "user-agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
        "referer": "https://pann.nate.com/"  # 요청의 출처를 표시하여 차단 우회
    },
    
    # BeautifulSoup 파서 설정
    bs_kwargs={
        # SoupStrainer: 특정 태그만 골라서 빠르게 파싱하도록 제한
        "parse_only": SoupStrainer(
            'dd',                          # dd 태그만 파싱
            attrs={'class': 'usertxt'}     # class="usertxt" 인 dd만 추출 (댓글 본문)
        )
    },
    
    # BeautifulSoup get_text() 옵션 설정
    bs_get_text_kwargs={
        "separator": "\n",  # 텍스트 사이 구분자: 여러 요소가 있으면 줄바꿈으로 구분
        "strip": True       # 앞뒤 공백 제거
    },
    
    # HTTP 요청 옵션 설정
    requests_kwargs={
        "timeout": 10,      # 요청 타임아웃 10초
        "verify": False     # SSL 인증서 검증 비활성화 (테스트 환경에서만 사용 권장)
    }
)

# 설정한 로더를 이용해 웹 페이지에서 문서(댓글들) 추출
docs = loader.load()

# 로드된 Document 객체 수 출력
print(f"로드된 문서의 수: {len(docs)}")

로드된 문서의 수: 1


#### PDFLoader

각 페이지를 `Document`로 변환

##### PyPDFLoader

- 가장 기본, 가볍고 빠름 -> 단순 텍스트 문서
- `extract_images`: 이미지 추출 여부
- `password`: 암호화된 pdf의 비밀번호

In [71]:
from langchain_community.document_loaders import PyPDFLoader

pdf_attention = "data/Attention Is All You Need.pdf"
pdf_bert = "data/BERT.pdf"
pdf_lg_aimers = "data/LG Aimers 4기 소개자료.pdf"

loader = PyPDFLoader(
    file_path=pdf_lg_aimers,
    extract_images=False)

docs = loader.load()
print(f"PDF 파일의 페이지 수: {len(docs)}")
print(docs[1].page_content)
docs[1].metadata

PDF 파일의 페이지 수: 7
1. LG Aimers 프로그램 개요
❑교육과 경험의 기회를 필요로 하는 청년들에게 양질의 AI교육을 온라인으로 제공하고, 기업의 실제 data를 다루며 
실무를 경험할 수 있는 기회를 제공하기 위해 『온라인 AI전문가 과정(1개월)』 과 『AI 해커톤 (Hackathon)(1개월)』을 
결합한 형태의 교육 프로그램
- 온라인 AI 교육 과정 : 국내 AI 전문가 (현업 전문가, 저명 교수)의 최신 AI 기법에 관한 온라인 강의(1개월)
• 온라인 초급 교육은 기초적인 프로그래밍데이터 처리에 관련된 내용들로써 외부 교육 콘텐츠를 활용(optional)
• 초급 교육은 자료구조와 알고리즘 등 컴퓨팅 관련 기본 소양 교육
• 온라인 중상급 교육은 국내 AI 분야 전문가들이 주제별로 강의 (LG 자체 개발- 부록 참조)
• 중상급 교육은 대학원 수준의 AI 요소 기술 및 이론에 관한 교육으로 이루어져 있음
• 중상급 교육을 받기 전에 학부 수준의 ‘인공지능 개론’ 과목을 이수하는 것을 추천
- 온라인 AI 해커톤 : LG계열사의 문제를 현장의 실제 data를 활용하여 해결하는 해커톤 예선(1개월)
- 오프라인 AI 해커톤 : 온라인 해커톤에서 선발된 본선 진출자들만 참가 (1박2일)
❑수료 조건 : 온라인 AI전문가 과정 이수(교육영상 100% 이수 및 퀴즈참여) & AI 해커톤 baseline model 성능 이상
수료자에 한하여 수료증 발급 
❑대상 및 규모 : 만 19세에서 29세의 청년 (미취업자 대상)
❑운영 일정 : 매년 연 2회 진행. 여름방학(7~8월), 겨울방학(1~2월)
❑LG Aimers 채널: https://lgaimers.ai/


{'producer': 'PyPDF',
 'creator': 'PyPDF',
 'creationdate': '2025-07-15T07:40:32+00:00',
 'moddate': '2025-07-15T07:40:32+00:00',
 'source': 'data/LG Aimers 4기 소개자료.pdf',
 'total_pages': 7,
 'page': 1,
 'page_label': '2'}

##### PDFPlumberLoader

- 표와 레이아웃 분석에 최적화된 로더 -> 형식이 있는 보고서나 논문에 적합
- `text_kwargs`: layout, tolerance, density
- `extract_images`: 이미지 추출 여부

In [70]:
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader(pdf_lg_aimers)
docs = loader.load()
print(f"PDF 파일의 페이지 수: {len(docs)}")

print(docs[1].page_content[:300])

PDF 파일의 페이지 수: 7
1. LG Aimers 프로그램 개요
❑교육과 경험의 기회를 필요로 하는 청년들에게 양질의 AI교육을 온라인으로 제공하고, 기업의 실제 data를 다루며
실무를 경험할 수 있는 기회를 제공하기 위해 『온라인 AI전문가 과정(1개월)』 과 『AI 해커톤 (Hackathon)(1개월)』을
결합한 형태의 교육 프로그램
- 온라인 AI 교육 과정 : 국내 AI 전문가 (현업 전문가, 저명 교수)의 최신 AI 기법에 관한 온라인 강의(1개월)
• 온라인 초급 교육은 기초적인 프로그래밍데이터 처리에 관련된 내용들로써 외부 교육 콘텐츠를 활용(


In [72]:
import pdfplumber

with pdfplumber.open(pdf_bert) as pdf:
    metadata = pdf.metadata
    print(metadata)

{'Author': '', 'CreationDate': 'D:20190528000751Z', 'Creator': 'LaTeX with hyperref package', 'Keywords': '', 'ModDate': 'D:20190528000751Z', 'PTEX.Fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) kpathsea version 6.2.2', 'Producer': 'pdfTeX-1.40.17', 'Subject': '', 'Title': '', 'Trapped': 'False'}


In [73]:
with pdfplumber.open(pdf_bert) as pdf:
    text = pdf.pages[0].extract_text()
    print(text[:300])

BERT: Pre-training of Deep Bidirectional Transformers for
Language Understanding
JacobDevlin Ming-WeiChang KentonLee KristinaToutanova
GoogleAILanguage
{jacobdevlin,mingweichang,kentonl,kristout}@google.com
Abstract There are two existing strategies for apply-
ingpre-trainedlanguage representations 


##### PyMuPDF4LLMLoader

- 마크다운 형식으로 변환하여 추출
- 이미지, 표, 복잡한 레이아웃 처리 가능
- `mode`: 페이지 단위로 문서화할지, 통째로 문서화할지 지정
- `extract_images`: 이미지 추출 여부
- `image_parser`: 추출된 이미지를 분석할 파서 지정, ex. LLMImageBlobParser

In [103]:
from langchain_pymupdf4llm import PyMuPDF4LLMLoader

pdf_lg_aimers = "data/LG Aimers 4기 소개자료.pdf"
txt_loader = PyMuPDF4LLMLoader(
    file_path=pdf_lg_aimers,
    mode="page"  # 페이지별로 분리
)

txt_docs = txt_loader.load()
print(f"PDF 파일의 페이지 수: {len(txt_docs)}")

# 텍스트 정보
print(txt_docs[1].page_content)

# 메타데이터
txt_docs[1].metadata

PDF 파일의 페이지 수: 7
#### **1. LG Aimers 프로그램개요**

###### ❑ 교육과 경험의 기회를 필요로 하는 청년들에게 양질의 AI교육을 온라인으로 제공하고, 기업의 실제 data를 다루며 실무를 경험할 수 있는 기회를 제공하기 위해 『온라인 AI전문가 과정(1개월)』과『AI 해커톤 (Hackathon) (1개월)』을 **결합한 형태의 교육 프로그램** **-** 온라인 AI 교육과정: 국내 AI 전문가 (현업 전문가, 저명 교수) 의 최신 AI 기법에관한 온라인 강의(1개월)


     **온라인 초급 교육은 기초적인 프로그래밍데이터 처리에 관련된 내용들로써 외부 교육 콘텐츠를 활용(optional)**


          **초급 교육은 자료구조와 알고리즘 등 컴퓨팅 관련 기본 소양 교육**


     **온라인 중상급 교육은 국내 AI 분야전문가들이 주제별로 강의 (LG 자체 개발-** **부록 참조)**


          **중상급 교육은 대학원 수준의 AI 요소 기술 및 이론에 관한 교육으로 이루어져 있음**


          **중상급 교육을 받기 전에 학부 수준의 ‘인공지능 개론’ 과목을 이수하는 것을 추천**

###### **-** **온라인 AI 해커톤: LG계열사의문제를 현장의 실제data를활용하여 해결하는 해커톤 예선(1개월)** **-** **오프라인 AI 해커톤: 온라인 해커톤에서 선발된 본선 진출자들만 참가 (1박2일)** ❑ 수료 조건 : 온라인 AI전문가 과정이수(교육영상100% 이수및퀴즈참여) & AI 해커톤 baseline model 성능이상 **수료자에 한하여 수료증 발급** ❑ 대상 및 규모 : 만 19세에서 29세의 청년 (미취업자 대상) ❑ 운영 일정 : 매년 연 2회 진행. 여름방학(7~8월), 겨울방학(1~2월)


❑ **LG Aimers 채널:** **[https://lgaimers.ai/](https://lgaimers.ai/)**





{'producer': '',
 'creator': '',
 'creationdate': '2025-07-15T07:40:32+00:00',
 'source': 'data/LG Aimers 4기 소개자료.pdf',
 'file_path': 'data/LG Aimers 4기 소개자료.pdf',
 'total_pages': 7,
 'format': 'PDF 1.7',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '2025-07-15T07:40:32+00:00',
 'trapped': '',
 'modDate': "D:20250715074032+00'00'",
 'creationDate': "D:20250715074032+00'00'",
 'page': 1}

> 이미지는? `fitz`로 추출해서 메타데이터에 추가해줘야 함!

- `fitz.open() -> Document`

In [104]:
from langchain_community.document_loaders.parsers import LLMImageBlobParser


img_loader = PyMuPDF4LLMLoader(
    pdf_lg_aimers,        # 로드할 PDF 파일 경로 또는 바이너리
    mode="page",          # 페이지 단위로 Document를 생성하도록 설정
    extract_images=True,  # PDF 내 포함된 이미지를 추출하도록 설정
    images_parser=LLMImageBlobParser(
        model=llm         # 추출된 이미지를 LLM 기반 파서로 분석
                          # (예: 이미지 캡션 생성, 이미지 내용 설명 등 가능)
    ),
)

NameError: name 'llm' is not defined

In [101]:
import fitz  # PyMuPDF
from pathlib import Path

# 이미지 저장 폴더 생성
output_dir = Path("extracted_images")
output_dir.mkdir(exist_ok=True)

# PDF 문서 열기
pdf_doc = fitz.open(pdf_lg_aimers)

# 각 페이지별로 이미지 추출 및 메타데이터 추가
for page_num in range(len(pdf_doc)):
    page = pdf_doc[page_num]
    image_list = page.get_images()
    
    # 해당 페이지의 이미지 파일명 리스트
    extracted_image_files = []
    
    # 페이지의 각 이미지 추출
    for img_index, img in enumerate(image_list):
        xref = img[0]  # 이미지 참조 번호
        
        try:
            # 이미지 추출
            base_image = pdf_doc.extract_image(xref)
            image_bytes = base_image["image"]
            image_ext = base_image["ext"]  # 이미지 확장자 (png, jpg 등)
            
            # 이미지 파일명 생성
            image_filename = f"page_{page_num}_img_{img_index}.{image_ext}"
            image_path = output_dir / image_filename
            
            # 이미지 저장
            with open(image_path, "wb") as img_file:
                img_file.write(image_bytes)
            
            extracted_image_files.append(str(image_path))
            
        except Exception as e:
            print(f"페이지 {page_num}, 이미지 {img_index} 추출 실패: {e}")
    
    # 해당 페이지의 Document 메타데이터에 이미지 파일명 추가
    if page_num < len(img_docs):
        img_docs[page_num].metadata['extracted_images'] = extracted_image_files
        img_docs[page_num].metadata['image_count'] = len(extracted_image_files)

pdf_doc.close()

print(f"이미지 추출 완료!")
print(f"저장 위치: {output_dir.absolute()}")

NameError: name 'img_docs' is not defined